In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

In [ ]:
data = pd.read_csv('California_Houses.csv')

Step 1: Split the dataset

In [ ]:
data=data.sample(frac=1, random_state=1).reset_index(drop=True)
n=len(data)
training_set = data[:int(n*0.7)]
validation_set = data[int(n*0.7) : int(n*0.85)]
test_set = data[int(n*0.85):]


Step 2: Obtain X and y

In [ ]:

y_train = training_set['Median_House_Value']
X_train = training_set.drop(columns=['Median_House_Value'])
y_val = validation_set['Median_House_Value']
X_val = validation_set.drop(columns=['Median_House_Value'])
y_test = test_set['Median_House_Value']
X_test = test_set.drop(columns=['Median_House_Value'])



Step 3: Scaling data

In [ ]:
min_val = X_train.min()
max_val = X_train.max()
X_train_s = (X_train - min_val) / (max_val - min_val)
X_val_s = (X_val - min_val) / (max_val - min_val)
X_test_s = (X_test - min_val) / (max_val - min_val)


Step 4: Add bias column

In [ ]:

X_train_sb = np.concatenate((np.ones((X_train_s.shape[0],1)), X_train_s.values),axis=1)
X_val_sb = np.concatenate((np.ones((X_val_s.shape[0],1)), X_val_s.values),axis=1)
X_test_sb = np.concatenate((np.ones((X_test_s.shape[0],1)), X_test_s.values),axis=1)


Implementing gradient descent:

In [ ]:
def gradient_descent(X, y, lr=0.06, iterations=10000, tol=1e-6):
    m, n = X.shape
    w = np.zeros(n)
    prev_loss = float('inf')

    for i in range(iterations):
        y_pred = X @ w
        error = y_pred - y
        loss=np.mean(error**2)
        if  abs(prev_loss - loss) < tol:
            print('loss converged at itteration ', i)
            break
        prev_loss = loss
        gradient = (1/m) * X.T @ error
        w = w - lr * gradient
    return w


In [ ]:
w = gradient_descent(X_train_sb, y_train.values)

Loss functions:


In [ ]:
def mse_loss(y_true, y_pred):
    return np.mean((y_true - y_pred)**2)
def mae_loss(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))




Make a predection


In [ ]:
y_val_predicted = X_val_sb @ w


Evaluation of the first model:

In [ ]:
print("MSE= ", mse_loss(y_val.values, y_val_predicted))
print("MAE= ", mae_loss(y_val.values, y_val_predicted))

The MAE indecates that the model predections deviate from the actual house prices by ~ $52k dollars on average. The MSE is large due to the squaring


**SKlearn**


In [190]:
model = LinearRegression()
model.fit(X_train_s, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](13,)","[ 561134.05, 45902.96,-194729.1 ,..., 272006.53, 152631.37,-148578.15]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](13,)","['Median_Income','Median_Age','Tot_Rooms',...,'Distance_to_SanDiego', 'Distance_to_SanJose','Distance_to_SanFrancisco']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,2.695e+05
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,13
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(13)


In [192]:
y_val_pred = model.predict(X_val_s)
y_test_pred = model.predict(X_test_s)

In [193]:
print("MSE= ", mse_loss(y_val.values, y_val_pred))
print("MAE= ", mae_loss(y_val.values, y_val_pred))

MSE=  4480379516.81691
MAE=  48963.255870880705


The MAE indecates that the model predections deviate from the actual house prices by ~ $49k dollars on average. The MSE is large due to the squaring


In [194]:
print("MSE= ", mse_loss(y_test.values, y_test_pred))
print("MAE= ", mae_loss(y_test.values, y_test_pred))

MSE=  4911048917.262813
MAE=  50802.801832641875


The sklearn implementation achieved results very similar to the closed form solution using the normal equation. It was the easiest approach.
Gradient descent required more careful tuning of learning rate and iterations count as I had to try different values to approach the the best results.